# Stage 2 — Genuineness

**Objective:** For each candidate group that survived Stage 1, test whether its intra-group similarity is statistically significant — higher than random groups of the same size.

**Method:** Permutation test (1000 iterations) at every layer where the group showed elevated similarity in Stage 1.

**Pass criterion:** Permutation p-value < 0.05 at the layers identified in Stage 1.

See `multicircuits.md` for full definitions and experiment plan.

In [ ]:
# Cell 1 – Setup & load atlas
import subprocess, sys, os, shutil
for pkg in ["h5py", "seaborn", "matplotlib", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)

DATASET = "115_strong"

_DATASETS = {
    "115_strong": "universal_115x100_strong",
    "115_medium": "universal_115x100_medium",
    "115_005": "universal_115x100_epsilon_005",
}
_base = _DATASETS[DATASET]

if IN_COLAB:
    DATA_DIR = "/content/drive/MyDrive/DATA/CSP-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/CSP-Atlas"

ATLAS_HDF5 = f"{DATA_DIR}/{_base}.h5"

LOCAL_SRC = "/Users/piotrwilam/Code/CSP-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/CSP-Atlas/src"
SRC_PATH  = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from module2.io_utils import load_atlas_hdf5

print(f"Environment : {'Colab' if IN_COLAB else 'Local'}")
print(f"Loading     : {ATLAS_HDF5}")
atlas = load_atlas_hdf5(ATLAS_HDF5)
pair_masks      = atlas["pair_masks"]
universal_masks = atlas["universal_masks"]

print(f"Pairs       : {len(pair_masks)}")
print(f"AST nodes   : {len(universal_masks['ast'])}")
print(f"Builtins    : {len(universal_masks['builtin'])}")

In [ ]:
# Cell 2 – Build mask matrix & define candidate groups
layer_ids = sorted({lid for lm in pair_masks.values() for lid in lm})

circuit_names = []
mask_rows = []
for name in sorted(universal_masks["ast"]):
    vec = [universal_masks["ast"][name].get(lid, np.zeros(2048)).astype(np.float32)
           for lid in layer_ids]
    circuit_names.append(f"AST:{name}")
    mask_rows.append(np.concatenate(vec))
for name in sorted(universal_masks["builtin"]):
    vec = [universal_masks["builtin"][name].get(lid, np.zeros(2048)).astype(np.float32)
           for lid in layer_ids]
    circuit_names.append(f"BLT:{name}")
    mask_rows.append(np.concatenate(vec))

mask_matrix = np.array(mask_rows)
n_circuits = len(circuit_names)
neurons_per_layer = mask_matrix.shape[1] // len(layer_ids)

# ── Candidate groups (same as 4A) ───────────────────────────────────────
GROUPS = {
    "A: Generic AST": {
        "AST:AsyncWith", "AST:Attribute", "AST:ClassDef", "AST:Delete",
        "AST:Dict", "AST:GeneratorExp", "AST:Lambda", "AST:Return",
        "AST:SetComp", "AST:Slice", "AST:Subscript", "AST:YieldFrom",
        "BLT:classmethod", "BLT:isinstance", "BLT:property", "BLT:repr",
        "BLT:staticmethod", "BLT:super", "BLT:zip",
    },
    "B: Exceptions": {
        "AST:Continue", "AST:DictComp", "AST:ListComp", "AST:With", "AST:Yield",
        "BLT:AttributeError", "BLT:Exception", "BLT:FileNotFoundError",
        "BLT:IndexError", "BLT:KeyError", "BLT:MemoryError", "BLT:NameError",
        "BLT:OSError", "BLT:OverflowError", "BLT:RecursionError",
        "BLT:RuntimeError", "BLT:StopIteration", "BLT:TypeError",
        "BLT:ValueError", "BLT:ZeroDivisionError",
    },
    "C: Raise+Errors": {
        "AST:While", "AST:Raise", "BLT:ArithmeticError",
        "BLT:ImportError", "BLT:LookupError", "BLT:NotImplementedError",
    },
    "D: Control flow": {
        "AST:For", "AST:AsyncFor", "AST:Global", "AST:ImportFrom",
        "AST:Nonlocal", "AST:Starred", "AST:ExceptHandler", "AST:Try",
        "AST:If", "AST:IfExp", "AST:FunctionDef", "AST:AsyncFunctionDef",
        "AST:Break",
    },
}

group_indices = {}
for gname, members in GROUPS.items():
    idx = [i for i, c in enumerate(circuit_names) if c in members]
    group_indices[gname] = idx
    print(f"{gname}: {len(idx)} circuits")

In [ ]:
# Cell 3 – Permutation test engine
N_PERMS = 1000
JACCARD_THRESHOLD = 0.1  # only test layers with intra-group J > this

def get_selective_layer(mask_matrix, layer_idx, neurons_per_layer, n_circuits):
    """Extract selective neuron submatrix for one layer."""
    s = layer_idx * neurons_per_layer
    e = s + neurons_per_layer
    layer_slice = mask_matrix[:, s:e]
    col_s = layer_slice.sum(axis=0)
    selective = (col_s > 0) & (col_s < n_circuits)
    return layer_slice[:, selective]

def mean_intra_jaccard(layer_sel, indices):
    """Mean pairwise Jaccard for a group on a selective-neuron submatrix."""
    if layer_sel.shape[1] == 0:
        return 0.0
    vals = []
    for ii in range(len(indices)):
        a = layer_sel[indices[ii]].astype(bool)
        for jj in range(ii + 1, len(indices)):
            b = layer_sel[indices[jj]].astype(bool)
            inter = (a & b).sum()
            union = (a | b).sum()
            vals.append(inter / union if union > 0 else 0.0)
    return np.mean(vals) if vals else 0.0

def permutation_test(layer_sel, group_indices, n_circuits, n_perms=1000):
    """
    Compare observed intra-group Jaccard against random groups of the same size.
    Returns (observed, p_value, perm_distribution).
    """
    k = len(group_indices)
    observed = mean_intra_jaccard(layer_sel, group_indices)

    np.random.seed(42)
    perm_vals = []
    for _ in range(n_perms):
        fake = list(np.random.choice(n_circuits, size=k, replace=False))
        perm_vals.append(mean_intra_jaccard(layer_sel, fake))

    perm_vals = np.array(perm_vals)
    p_value = (perm_vals >= observed).mean()
    return observed, p_value, perm_vals

print(f"Permutation test engine ready | N_PERMS={N_PERMS}")

In [ ]:
# Cell 4 – Run permutation tests for all groups x all layers
all_results = []

for gname, indices in group_indices.items():
    print(f"\n{'='*60}")
    print(f"{gname} ({len(indices)} circuits)")
    print(f"{'='*60}")

    for li, lid in enumerate(layer_ids):
        layer_sel = get_selective_layer(mask_matrix, li, neurons_per_layer, n_circuits)
        n_sel = layer_sel.shape[1]

        if n_sel == 0:
            all_results.append({
                "group": gname, "layer": lid, "n_selective": 0,
                "observed": 0.0, "p_value": 1.0, "perm_mean": 0.0,
                "perm_95": 0.0, "significant": False, "tested": False,
            })
            continue

        observed = mean_intra_jaccard(layer_sel, indices)

        # Only run full permutation test if above threshold
        if observed <= JACCARD_THRESHOLD:
            all_results.append({
                "group": gname, "layer": lid, "n_selective": n_sel,
                "observed": observed, "p_value": 1.0, "perm_mean": 0.0,
                "perm_95": 0.0, "significant": False, "tested": False,
            })
            continue

        obs, p_val, perm_dist = permutation_test(
            layer_sel, indices, n_circuits, n_perms=N_PERMS)

        sig = p_val < 0.05
        all_results.append({
            "group": gname, "layer": lid, "n_selective": n_sel,
            "observed": obs, "p_value": p_val,
            "perm_mean": perm_dist.mean(), "perm_95": np.percentile(perm_dist, 95),
            "significant": sig, "tested": True,
        })

        status = "SIGNIFICANT" if sig else "not significant"
        print(f"  Layer {lid}: J={obs:.4f} | p={p_val:.4f} | "
              f"random={perm_dist.mean():.4f} | {status}")

results_df = pd.DataFrame(all_results)
print("\n\nFull results table:")
display(results_df[results_df["tested"]][
    ["group", "layer", "n_selective", "observed", "p_value", "perm_mean", "significant"]
])

In [ ]:
# Cell 5 – Permutation histograms for tested layers
import matplotlib.pyplot as plt

colors = {"A: Generic AST": "#2c7bb6", "B: Exceptions": "#d7191c",
          "C: Raise+Errors": "#fdae61", "D: Control flow": "#abd9e9"}

tested = results_df[results_df["tested"]].copy()

for gname in GROUPS:
    g_tested = tested[tested["group"] == gname]
    if g_tested.empty:
        print(f"{gname}: no layers tested (all below J threshold)")
        continue

    n_plots = len(g_tested)
    n_cols = min(2, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4 * n_rows))

    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes[np.newaxis, :]
    elif n_cols == 1:
        axes = axes[:, np.newaxis]

    for plot_idx, (_, row) in enumerate(g_tested.iterrows()):
        r, c = divmod(plot_idx, n_cols)
        ax = axes[r, c]
        lid = int(row["layer"])

        # Rerun to get distribution for plotting
        li = layer_ids.index(lid)
        layer_sel = get_selective_layer(mask_matrix, li, neurons_per_layer, n_circuits)
        _, _, perm_dist = permutation_test(
            layer_sel, group_indices[gname], n_circuits, n_perms=N_PERMS)

        ax.hist(perm_dist, bins=40, edgecolor="black", alpha=0.7,
                color="#cccccc", label="Random")
        ax.axvline(row["observed"], color=colors[gname], linewidth=2,
                   linestyle="--", label=f"Observed = {row['observed']:.4f}")
        ax.axvline(np.percentile(perm_dist, 95), color="orange", linewidth=1,
                   linestyle=":", label=f"95th pct = {np.percentile(perm_dist, 95):.4f}")

        sig = "p < 0.05" if row["significant"] else f"p = {row['p_value']:.3f}"
        ax.set_title(f"Layer {lid} — {sig}")
        ax.set_xlabel("Mean intra-group Jaccard")
        ax.set_ylabel("Count")
        ax.legend(fontsize=7)

    for plot_idx in range(n_plots, n_rows * n_cols):
        r, c = divmod(plot_idx, n_cols)
        axes[r, c].set_visible(False)

    fig.suptitle(f"{gname} — Permutation tests", fontsize=13)
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 6 – Summary heatmap: p-values per group x layer
pivot_p = results_df.pivot(index="group", columns="layer", values="p_value")
pivot_obs = results_df.pivot(index="group", columns="layer", values="observed")
pivot_tested = results_df.pivot(index="group", columns="layer", values="tested")

# Annotate: show observed J where tested, "—" otherwise
annot = pivot_obs.copy().astype(str)
for gname in GROUPS:
    for lid in layer_ids:
        t = pivot_tested.loc[gname, lid]
        if not t:
            annot.loc[gname, lid] = "—"
        else:
            obs = pivot_obs.loc[gname, lid]
            p = pivot_p.loc[gname, lid]
            star = "*" if p < 0.05 else ""
            annot.loc[gname, lid] = f"{obs:.3f}{star}"

import matplotlib.colors as mcolors

# p-value colormap: green (significant) to white (not)
fig, ax = plt.subplots(figsize=(12, 4))
p_display = pivot_p.fillna(1.0).values
sns.heatmap(p_display, ax=ax, vmin=0, vmax=1, cmap="RdYlGn_r",
            xticklabels=layer_ids, yticklabels=list(GROUPS.keys()),
            annot=annot.values, fmt="", annot_kws={"fontsize": 9},
            cbar_kws={"label": "p-value"})
ax.set_title("Stage 2: Permutation p-values (* = significant at p < 0.05)\n"
             "Cell values = observed mean Jaccard")
ax.set_xlabel("Layer")
plt.tight_layout(); plt.show()

In [ ]:
# Cell 7 – Verdict
MIN_SIG_LAYERS = 1  # must be significant at at least this many layers

print("=" * 65)
print("STAGE 2 VERDICT — GENUINENESS")
print("=" * 65)

survivors = []
for gname in GROUPS:
    g_data = results_df[(results_df["group"] == gname) & results_df["tested"]]
    sig_layers = g_data[g_data["significant"]]
    n_sig = len(sig_layers)
    layers = sig_layers["layer"].tolist()

    passed = n_sig >= MIN_SIG_LAYERS
    status = "PASS" if passed else "FAIL"
    if passed:
        survivors.append(gname)

    print(f"\n{gname}")
    print(f"  Layers tested       : {len(g_data)}")
    print(f"  Significant (p<0.05): {n_sig} — {layers}")
    if not sig_layers.empty:
        best = sig_layers.loc[sig_layers["observed"].idxmax()]
        print(f"  Best layer          : {int(best['layer'])} "
              f"(J={best['observed']:.4f}, p={best['p_value']:.4f})")
    print(f"  Verdict             : {status}")

print(f"\n{'='*65}")
print(f"\nGenuine multicircuits: {survivors if survivors else 'NONE'}")
print(f"Proceed to Stage 3 (epsilon robustness) with these groups.")

## Leave-One-Out Refinement

For groups A and D that failed or barely passed, try removing one member at a time to find which circuits are dragging the group down. The goal: find the genuine core subgroup.

In [ ]:
# Cell 9 – Leave-one-out analysis for groups A and D
REFINE_GROUPS = ["A: Generic AST", "D: Control flow"]

def leave_one_out(indices, layer_sels):
    """For each member, compute mean Jaccard across layers with it removed."""
    results = []
    full_mean = np.mean([mean_intra_jaccard(ls, indices) for ls in layer_sels])

    for drop_pos in range(len(indices)):
        reduced = indices[:drop_pos] + indices[drop_pos+1:]
        reduced_mean = np.mean([mean_intra_jaccard(ls, reduced) for ls in layer_sels])
        delta = reduced_mean - full_mean
        results.append({
            "dropped": circuit_names[indices[drop_pos]],
            "remaining_mean_J": reduced_mean,
            "delta": delta,
        })
    return pd.DataFrame(results)

# Precompute selective layer matrices
layer_sels = []
for li in range(len(layer_ids)):
    ls = get_selective_layer(mask_matrix, li, neurons_per_layer, n_circuits)
    layer_sels.append(ls)

# Store results for report
loo_results = {}

for gname in REFINE_GROUPS:
    indices = group_indices[gname]
    baseline = np.mean([mean_intra_jaccard(ls, indices) for ls in layer_sels])

    loo_df = leave_one_out(indices, layer_sels)
    loo_df = loo_df.sort_values("delta", ascending=False).reset_index(drop=True)
    loo_results[gname] = {"baseline": baseline, "table": loo_df}

    print(f"\n{'='*65}")
    print(f"{gname} — Leave-One-Out (baseline mean J across layers: {baseline:.4f})")
    print(f"{'='*65}")
    print(f"Positive delta = removing this member IMPROVES group cohesion\n")
    display(loo_df)

In [ ]:
# Cell 10 – Iterative pruning: keep dropping worst member until group is genuine

# Store results for report
pruning_results = {}

for gname in REFINE_GROUPS:
    print(f"\n{'='*65}")
    print(f"{gname} — Iterative pruning")
    print(f"{'='*65}")

    current_indices = list(group_indices[gname])
    dropped = []
    history = []

    for iteration in range(len(current_indices) - 3):  # keep at least 3 members
        # Leave-one-out: find member whose removal maximizes mean J
        best_reduced_mean = -1
        best_drop = 0
        for drop_pos in range(len(current_indices)):
            reduced = current_indices[:drop_pos] + current_indices[drop_pos+1:]
            reduced_mean = np.mean([mean_intra_jaccard(ls, reduced) for ls in layer_sels])
            if reduced_mean > best_reduced_mean:
                best_reduced_mean = reduced_mean
                best_drop = drop_pos

        # Compute current stats
        current_mean = np.mean([mean_intra_jaccard(ls, current_indices) for ls in layer_sels])

        # Count significant layers
        n_sig = 0
        for ls in layer_sels:
            if ls.shape[1] == 0:
                continue
            obs = mean_intra_jaccard(ls, current_indices)
            if obs <= JACCARD_THRESHOLD:
                continue
            _, p, _ = permutation_test(ls, current_indices, n_circuits, n_perms=200)
            if p < 0.05:
                n_sig += 1

        history.append({
            "iteration": iteration,
            "size": len(current_indices),
            "mean_J": current_mean,
            "sig_layers": n_sig,
            "next_drop": circuit_names[current_indices[best_drop]],
        })

        print(f"  Step {iteration}: {len(current_indices)} members | "
              f"mean J={current_mean:.4f} | sig layers={n_sig} | "
              f"drop → {circuit_names[current_indices[best_drop]]}")

        # If already significant at 3+ layers, stop
        if n_sig >= 3 and iteration > 0:
            print(f"\n  CONVERGED: {len(current_indices)} members, {n_sig} significant layers")
            break

        # Drop worst member
        dropped.append(circuit_names[current_indices[best_drop]])
        current_indices.pop(best_drop)

    # Final group
    final_names = sorted([circuit_names[i] for i in current_indices])
    print(f"\n  Refined group ({len(final_names)}):")
    for name in final_names:
        print(f"    {name}")
    print(f"  Dropped ({len(dropped)}):")
    for name in dropped:
        print(f"    {name}")

    history_df = pd.DataFrame(history)
    display(history_df)

    pruning_results[gname] = {
        "final_members": final_names,
        "dropped": dropped,
        "history": history_df,
        "final_size": len(final_names),
        "final_sig_layers": history[-1]["sig_layers"] if history else 0,
        "final_mean_J": history[-1]["mean_J"] if history else 0,
    }

In [ ]:
# Cell 11 – Export report
import datetime

report_dir = DATA_DIR if IN_COLAB else "/Users/piotrwilam/Data/CSP-Atlas"
report_path = os.path.join(report_dir, f"report_4B_{DATASET}_{datetime.date.today()}.txt")

_meta = atlas.get("metadata", {})

with open(report_path, "w") as f:
    f.write("STAGE 2 — GENUINENESS REPORT\n")
    f.write(f"Generated: {datetime.datetime.now().isoformat()}\n")
    f.write(f"Dataset: {DATASET} ({ATLAS_HDF5})\n")
    f.write(f"Circuits: {n_circuits} | Layers: {layer_ids}\n")
    f.write(f"Epsilon: {_meta.get('epsilon', '?')} | "
            f"Consistency: {_meta.get('consistency_thresh', '?')}\n")
    f.write(f"Permutations: {N_PERMS} | Jaccard threshold: {JACCARD_THRESHOLD}\n")
    f.write(f"\n{'='*65}\n")

    # Full results table
    f.write("\nPermutation test results (tested layers only):\n")
    tested_cols = ["group", "layer", "n_selective", "observed", "p_value", "perm_mean", "significant"]
    f.write(results_df[results_df["tested"]][tested_cols].to_string(index=False))
    f.write("\n")

    # Per-group detail
    f.write(f"\n{'='*65}\n")
    f.write("VERDICT\n")
    f.write(f"{'='*65}\n")
    for gname in GROUPS:
        g_data = results_df[(results_df["group"] == gname) & results_df["tested"]]
        sig_layers = g_data[g_data["significant"]]
        n_sig = len(sig_layers)
        layers = sig_layers["layer"].tolist()
        passed = n_sig >= MIN_SIG_LAYERS

        f.write(f"\n{gname} ({len(group_indices[gname])} circuits)\n")
        f.write(f"  Members: {sorted(GROUPS[gname])}\n")
        f.write(f"  Layers tested       : {len(g_data)}\n")
        f.write(f"  Significant (p<0.05): {n_sig} — {layers}\n")
        if not sig_layers.empty:
            best = sig_layers.loc[sig_layers["observed"].idxmax()]
            f.write(f"  Best layer          : {int(best['layer'])} "
                    f"(J={best['observed']:.4f}, p={best['p_value']:.4f})\n")
        f.write(f"  Verdict             : {'PASS' if passed else 'FAIL'}\n")

    f.write(f"\n{'='*65}\n")
    f.write(f"Genuine multicircuits: {survivors if survivors else 'NONE'}\n")

    # Leave-one-out results
    if loo_results:
        f.write(f"\n{'='*65}\n")
        f.write("LEAVE-ONE-OUT ANALYSIS\n")
        f.write(f"{'='*65}\n")
        for gname, res in loo_results.items():
            f.write(f"\n{gname} (baseline mean J: {res['baseline']:.4f})\n")
            f.write(res["table"].to_string(index=False))
            f.write("\n")

    # Iterative pruning results
    if pruning_results:
        f.write(f"\n{'='*65}\n")
        f.write("ITERATIVE PRUNING\n")
        f.write(f"{'='*65}\n")
        for gname, res in pruning_results.items():
            f.write(f"\n{gname}\n")
            f.write(f"  Final size       : {res['final_size']}\n")
            f.write(f"  Final mean J     : {res['final_mean_J']:.4f}\n")
            f.write(f"  Final sig layers : {res['final_sig_layers']}\n")
            f.write(f"  Final members    : {res['final_members']}\n")
            f.write(f"  Dropped          : {res['dropped']}\n")
            f.write(f"\n  Pruning history:\n")
            f.write(res["history"].to_string(index=False))
            f.write("\n")

print(f"Report saved: {report_path}")